# Topic 02 — Visual Analysis Toolbox

This notebook is a compact map from **analytical question → chart → interpretation**. The goal is to make plotting decisions explicit instead of drawing charts mechanically.

> Reasoning blocks below are concise, reviewable explanations of the analytical approach: what I want to test, why the chart fits, and what conclusion is justified.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/'
df = pd.read_csv(DATA_URL + 'telecom_churn.csv')
df.shape

## 1. One numerical feature → histogram + KDE

**Question.** What does the distribution of daytime call duration look like?

**Why this chart.** A histogram shows mass in intervals; KDE adds a smoothed view of the shape. I am looking for skewness, multiple modes and suspicious tails.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['Total day minutes'], bins=30, kde=True, ax=ax)
ax.set_title('Distribution of total day minutes')
plt.show()
df['Total day minutes'].describe()

**Interpretation.** The chart gives the shape, while `describe()` anchors the impression with exact quartiles and range. A smooth bell-like shape does not prove normality, but it is a useful first diagnostic.

## 2. Numerical feature + possible outliers → boxplot

**Question.** Are unusually high numbers of international calls common?

**Why this chart.** A boxplot compresses median, IQR and potential outliers into one view.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
sns.boxplot(x=df['Total intl calls'], ax=ax)
ax.set_title('International calls: spread and unusual values')
plt.show()

**Caution.** A point beyond a whisker is not automatically a bad record. It is a candidate for investigation, not a deletion command.

## 3. Categorical feature → countplot

**Question.** Is the target balanced?

**Why this chart.** For categories, counts are more informative than forcing a numerical distribution plot.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', ax=ax)
ax.set_title('Target class balance')
plt.show()
df['Churn'].value_counts(normalize=True).rename('share')

**Interpretation.** The numerical share is the important companion to the visual impression. Class imbalance will matter later when choosing model metrics.

## 4. Numerical feature split by target → box/violin plot

**Question.** Do churned and retained clients have visibly different daytime usage?

**Why this chart.** We need to compare distributions across two groups, not only their means.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x='Churn', y='Total day minutes', ax=axes[0])
sns.violinplot(data=df, x='Churn', y='Total day minutes', ax=axes[1], inner='quartile')
axes[0].set_title('Boxplot')
axes[1].set_title('Violin plot')
plt.tight_layout()
plt.show()
df.groupby('Churn')['Total day minutes'].agg(['mean', 'median', 'std'])

## 5. Two numerical features → scatterplot

**Question.** Are day minutes and day charge effectively the same information?

**Why this chart.** A scatterplot reveals functional or approximately linear relationships between two numerical variables.

In [ ]:
sample = df.sample(min(1200, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=sample, x='Total day minutes', y='Total day charge', hue='Churn', alpha=.6, ax=ax)
ax.set_title('Day minutes vs day charge')
plt.show()
df[['Total day minutes', 'Total day charge']].corr()

**Interpretation.** A near-perfect relationship suggests redundancy. Correlation is useful here because the visual hypothesis is explicitly linear.

## 6. Many numerical relationships → correlation heatmap

**Question.** Which numerical features move together strongly enough to deserve closer inspection?

**Why this chart.** A heatmap is a compact screening tool. It is not evidence of causality.

In [ ]:
numeric = df.select_dtypes(include='number')
corr = numeric.corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Pearson correlation matrix')
plt.tight_layout()
plt.show()

## Decision rule I want to remember

| Question | First chart to try |
|---|---|
| What is the shape of one numeric feature? | histogram / KDE |
| Are there unusual values? | boxplot |
| How frequent are categories? | countplot |
| How does a numeric feature differ by group? | box / violin / grouped histogram |
| How do two numeric features relate? | scatterplot |
| Which numeric pairs move together? | correlation heatmap |

The chart is only the start. I should verify important visual impressions with a numerical summary before turning them into a conclusion.